# Pose Control based on Lyapunov Estabilization theory

- Como se faz um projeto não-linear
- A chegada na equação de Lyapunov

## Configuring the environment

In [75]:
import numpy as np
import math
import matplotlib.pyplot as plt
from coppeliasim_zmqremoteapi_client import RemoteAPIClient
import os

# Parameters of Turtlebot 3
wheel_radius = 0.033
robot_width = 0.287

# Initialize the Remote API
client = RemoteAPIClient()
sim = client.require('sim')

# Open the turtlebot3 scene 
# simulation_file = os.getcwd()+'/turtlebot3_pose_estabilization.ttt'
# sim.loadScene(simulation_file)

# Use the stepping mode
sim.setStepping(True)   

# Configure the handles
Turtlebot3 = sim.getObjectHandle('/Turtlebot3')
leftMotor = sim.getObjectHandle('/Turtlebot3/left_motor')
rightMotor = sim.getObjectHandle('/Turtlebot3/right_motor')
Goal = sim.getObjectHandle('/ReferenceFrame')

# Error tolerance
min_error = 0.05

# Robot Limits - considering the kinematic model
phi_wheel_max = 3.5

In [76]:
def draw_robot(x, y, theta, ax, scale=1.0, **kwargs):
    """
    Desenha o contorno do robô no gráfico fornecido.
    
    Parâmetros:
    x, y  : Posição do centro do robô (m)
    theta : Orientação do robô (rad)
    ax    : Objeto Axes do matplotlib onde o robô será desenhado
    scale : Escala do desenho (para ajustar o tamanho visualmente)
    **kwargs: Argumentos opcionais de plotagem (ex: color='b', linestyle='--')
    """
    p = np.zeros((12, 3))
    
    p[:] = [
        [ 1,    1/7,  1/scale],
        [-3/7,  1,    1/scale],
        [-5/7,  6/7,  1/scale],
        [-5/7,  5/7,  1/scale],
        [-3/7,  2/7,  1/scale],
        [-3/7,  0,    1/scale],
        [-3/7, -2/7,  1/scale],
        [-5/7, -5/7,  1/scale],
        [-5/7, -6/7,  1/scale],
        [-3/7, -1,    1/scale],
        [ 1,   -1/7,  1/scale],
        [ 1,    1/7,  1/scale]
    ]
    
    p = scale * p
    
    r = np.array([
        [np.cos(theta),  np.sin(theta)],
        [-np.sin(theta), np.cos(theta)],
        [x,              y]
    ])
    
    p_transf = np.dot(p, r)
    
    X_plot = p_transf[:, 0]
    Y_plot = p_transf[:, 1]
    
    if 'color' not in kwargs and 'c' not in kwargs:
        kwargs['color'] = 'blue'
        
    ax.plot(X_plot, Y_plot, **kwargs)

# Normalize angle to the range [-pi,pi)
def normalize_angle(angle):
    return np.mod(angle+np.pi, 2*np.pi) - np.pi



## Simulation Loop - Aicardi

In [77]:
# Initialize the simulation
sim.startSimulation()

# Control gains
gamma = 0.5
h = 2.0
k = 2.0

while True:

    # Capture the robot pose from simulation
    TBPos=sim.getObjectPosition(Turtlebot3,-1)
    TBXOri=sim.getObjectOrientation(Turtlebot3,-1)
    qTurtlebot = np.array([TBPos[0], TBPos[1], TBXOri[2]])

    # Capture the goal pose
    Goal_Pos = sim.getObjectPosition(Goal,-1)
    Goal_Ori = sim.getObjectOrientation(Goal,-1)
    qGoal = np.array([Goal_Pos[0], Goal_Pos[1],Goal_Ori[2]])

    # Global States
    dx, dy, dth = qGoal - qTurtlebot

    # Transform to Aicardi states
    e = math.sqrt(dx**2 + dy**2)
    alpha = normalize_angle(np.arctan2(dy,dx) - qTurtlebot[2])
    theta_aicardi = normalize_angle(qGoal[2] - np.arctan2(dy,dx))

    # Stopping condition
    if e < min_error:
        # Goal achieved
        print("Alvo alcançado!")
        sim.stopSimulation()
        break

    # Calculate linear and angular velocities from (6) and (9)
    # u = gamma * cos(alpha) * e
    v = gamma * math.cos(alpha) * e
    
    # omega = k*alpha + gamma * (cos(alpha)*sin(alpha)/alpha) * (alpha + h*theta)
    omega = k * alpha + gamma * ((math.cos(alpha) * math.sin(alpha))/alpha) * (alpha + h * theta_aicardi)

    #  DDMR Inverse Kinematics
    w_right = (v + (omega * robot_width / 2)) / wheel_radius
    w_left  = (v - (omega * robot_width / 2)) / wheel_radius

    # Saturation Limits
    # w_right = max(min(w_right, phi_wheel_max), -phi_wheel_max)
    # w_left = max(min(w_left, phi_wheel_max), -phi_wheel_max)

    # Send commands
    sim.setJointTargetVelocity(leftMotor, w_left)
    sim.setJointTargetVelocity(rightMotor, w_right)

    print(f"Erro: {e} | Alpha: {alpha} | Theta: {theta_aicardi}")
    print(f"v: {v}, omega: {omega}")
    # Step the simulation
    sim.step()

Erro: 2.3053045351970303 | Alpha: -0.6954047576309277 | Theta: -0.1772598683662956
v: 0.8849999999999996, omega: -1.7621630540685715
Erro: 2.2962713869248637 | Alpha: -0.6991209929518836 | Theta: -0.1739497402556296
v: 0.8787924327199238, omega: -1.7670878668143946
Erro: 2.2722092105011313 | Alpha: -0.6862851049647336 | Theta: -0.16418235914235346
v: 0.8788966082947202, omega: -1.7349489017615
Erro: 2.2414776779450967 | Alpha: -0.6146598098994764 | Theta: -0.1504904170235979
v: 0.9156096452733478, omega: -1.5802343108246517
Erro: 2.2068469490773674 | Alpha: -0.5488271338504216 | Theta: -0.13725213188215957
v: 0.9413713681954564, omega: -1.4314938781943125
Erro: 2.1689720734474602 | Alpha: -0.489896752540246 | Theta: -0.12468421578310007
v: 0.9569303559872693, omega: -1.2930598939795674
Erro: 2.128628579308677 | Alpha: -0.43721684400603555 | Theta: -0.11293756788063858
v: 0.9641980904147978, omega: -1.1653140375757012
Erro: 2.086475807217363 | Alpha: -0.3900958086760711 | Theta: -0.1020